In [ ]:
from pathlib import Path
import os

repo_dir = Path.cwd() / 'Cradle'
if not (repo_dir / 'README.md').exists():
    !git clone https://github.com/BAAI-Agents/Cradle.git {repo_dir}
os.chdir(repo_dir)

# Gemma 4 E4B GTA SA Cloud Training

This notebook is the cloud-first version of the GTA SA vision fine-tuning flow.

It assumes the raw session tree is available on the cloud machine as a local folder or a mounted disk. The mapper and converter both run against that local copy so the image paths stay valid for training.

If you start from Google Cloud Storage, copy the full `runs/<date>/time_start_<time>/` folder down first, then keep that directory structure intact. The training JSONL will contain local file paths, not `gs://` URIs.

## Installation

In [ ]:
%%capture
import os, re
if 'COLAB_' not in ''.join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10': '0.0.34', '2.9': '0.0.33.post1', '2.8': '0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf 'datasets==4.3.0' 'huggingface_hub>=0.34.0' hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
!pip install --no-deps --upgrade timm
import torch
torch._dynamo.config.recompile_limit = 64

## Cloud Session Setup

Set `CLOUD_SESSION_URI` to the raw session folder you copied to cloud storage. If the session is already mounted locally, leave `SYNC_FROM_GCS` as `False` and point `LOCAL_SESSION_DIR` at the mounted path.

In [ ]:
from pathlib import Path

CLOUD_SESSION_URI = 'gs://YOUR_BUCKET/runs/2026-04-15/time_start_00-48-25-259432'
LOCAL_DATA_ROOT = Path.cwd() / 'cradle-data'
LOCAL_SESSION_DIR = LOCAL_DATA_ROOT / 'runs' / '2026-04-15' / 'time_start_00-48-25-259432'
MAPPED_DIR = LOCAL_SESSION_DIR / 'mapped_dataset'
MAPPED_JSONL = MAPPED_DIR / 'mapped_frames.jsonl'
GEMMA_JSONL = MAPPED_DIR / 'gemma_train.jsonl'
SYNC_FROM_GCS = False

LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
if SYNC_FROM_GCS and not LOCAL_SESSION_DIR.exists():
    !gsutil -m cp -r {CLOUD_SESSION_URI} {LOCAL_SESSION_DIR.parent}

In [ ]:
required_paths = [
    LOCAL_SESSION_DIR / 'manifest.json',
    LOCAL_SESSION_DIR / 'frame_timeline.jsonl',
    LOCAL_SESSION_DIR / 'inputs.jsonl',
    LOCAL_SESSION_DIR / 'video.mp4',
]

missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('\n'.join(missing_paths))

!python tools/map_gta_sa_session.py --session-dir {LOCAL_SESSION_DIR} --output-dir {MAPPED_DIR}
!python tools/convert_mapped_frames_to_gemma_jsonl.py --input {MAPPED_JSONL} --output {GEMMA_JSONL} --image-root {MAPPED_DIR}

## Dataset Loading

The converted JSONL is ready for vision fine-tuning. Each row contains a user message with an image placeholder and the resolved local frame path under `images`.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=str(GEMMA_JSONL), split='train')
dataset = dataset.shuffle(seed=3407)
splits = dataset.train_test_split(test_size=0.02, seed=3407)
train_dataset = splits['train']
eval_dataset = splits['test']

len(train_dataset), len(eval_dataset)

## Model Setup

This loads Gemma 4 E4B in 4-bit mode and attaches LoRA adapters for the full vision-language stack.

In [ ]:
from unsloth import FastVisionModel, get_chat_template
import torch

model, processor = FastVisionModel.from_pretrained(
    'unsloth/gemma-4-E4B-it',
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)

processor = get_chat_template(processor, 'gemma-4')

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
    target_modules='all-linear',
)

## Training

The default config is conservative enough for a cloud L4-style runtime while still being easy to scale up if you have a larger GPU.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor),
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=5,
        save_strategy='epoch',
        optim='adamw_8bit',
        weight_decay=0.001,
        lr_scheduler_type='cosine',
        seed=3407,
        output_dir='outputs/gemma4_gta_sa_cloud',
        report_to='none',
        remove_unused_columns=False,
        dataset_text_field='',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_length=2048,
    ),
)

trainer_stats = trainer.train()

## Smoke Test and Save

Run one inference example after training, then save the LoRA adapters locally. You can upload the saved folder to cloud storage afterward if needed.

In [ ]:
from PIL import Image
from transformers import TextStreamer

sample = eval_dataset[0]
image = Image.open(sample['images'][0]).convert('RGB')
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': 'Predict the next 6 actions.'},
        ],
    }
]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors='pt',
).to('cuda')

text_streamer = TextStreamer(processor.tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
)

model.save_pretrained('gemma_4_lora_cloud')
processor.save_pretrained('gemma_4_lora_cloud')